# Phase 4 — Agent graph

Runs 5 mixed questions through the full LangGraph pipeline: scope -> planner -> router -> answer_steps -> verify. Requires `GROQ_API_KEY` (Kaggle Secret) plus the ingested corpus from Phase 1.

In [ ]:
import subprocess, sys, os
REPO_URL = "https://github.com/<your-username>/stance.git"  # TODO: set this
if not os.path.isdir("/kaggle/working/stance"):
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/stance"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/stance[retrieval,agents]"], check=True)
sys.path.insert(0, "/kaggle/working/stance")

In [ ]:
from stance.db import get_engine
from stance.agents.graph import build_graph
import json

engine = get_engine()
graph = build_graph(engine, latest_available_year=2024)

In [ ]:
TEST_QUESTIONS = [
    "What was Apple's FY24 revenue?",                       # numeric, fast path
    "What was Apple's revenue growth from FY23 to FY24?",     # numeric computation, fast path
    "What are Apple's main risk factors?",                    # narrative, full path
    "Compare Apple and Microsoft FY24 revenue",                # multi-company numeric, full path
    "What were Delta's risk factors?",                          # unresolvable entity -> clarification
]

for i, question in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*80}\nQ{i+1}: {question}\n{'='*80}")
    result = graph.invoke({"question": question}, config={"configurable": {"thread_id": f"demo-{i}"}})

    if "clarification" in result:
        print(f"CLARIFICATION NEEDED: {result['clarification']}")
        continue

    print(f"Route: {result.get('route')}")
    print(f"Answer: {result.get('final_answer')}")
    print(f"Verifier report: {json.dumps(result.get('verifier_report'), indent=2)}")

## Exit criteria check

- Q1, Q2 should route `fast` (single-company, single/two-year numeric).
- Q3, Q4 should route `full` (narrative / multi-company).
- Q5 should stop at clarification, never guessing a company.

In [ ]:
# Checkpointing check: resuming a thread should not re-run completed nodes.
# (LangGraph's MemorySaver keeps state per thread_id; invoking again with the
# same thread_id and no new input returns the cached final state.)
resumed = graph.invoke(None, config={"configurable": {"thread_id": "demo-0"}})
print("Resumed state matches original:", resumed.get("final_answer") is not None)